# 2. Surface-temperature grid

Runs the solver over the temperature range in `run_toi561b.yaml` (`grid:` section). Temperatures that already have a converged output file are skipped.

In [ ]:
from gce_tools import RunConfig, run_grid

cfg = RunConfig.from_yaml("run_toi561b.yaml")
print(cfg.network.summary())
print("Temperatures:", cfg.grid.temperatures)
print("Output dir:  ", cfg.run_dir)

In [ ]:
results = run_grid(cfg)
results

## Quick look

In [ ]:
import matplotlib.pyplot as plt

gas = cfg.network.phases["gas"]
fig, ax = plt.subplots()
for sp in gas.species:
    ax.semilogy(results["T_surface"], results[gas.column(sp)], label=sp)
ax.set_xlabel("Surface temperature [K]")
ax.set_ylabel("Gas mole fraction")
ax.set_ylim(1e-10, 1.5)
ax.legend(ncol=2, fontsize=8)
plt.show()

## Mass conservation check
**Left:** total planet mass from the converged element totals, as a fractional deviation from the nominal mass in the run YAML. This should stay at round-off level (around 1e-10 or smaller) at every temperature.

**Right:** surface pressure from the converged atmospheric mass, P = M_atm g / 4πR². Unlike the total mass, this is free to drift away from the input pressure (dashed line), because outgassing and dissolution move mass between the atmosphere and the melt.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

T = results["T_surface"]
dM = results["M_total_over_M_planet"] - 1.0
P = results["P_surface_bar"]
P0 = cfg.planet.surface_pressure_bar

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

ax1.plot(T, dM, "o-")
ax1.axhline(0, color="k", lw=0.8, ls="--")
ax1.set_xlabel("Surface temperature [K]")
ax1.set_ylabel(r"$M_{\rm total}/M_{\rm planet} - 1$")
ax1.set_title(f"Total mass (max |dev| = {np.abs(dM).max():.2e})")
ax1.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))

ax2.plot(T, P, "o-", label="converged")
ax2.axhline(P0, color="k", lw=0.8, ls="--", label=f"input ({P0:.0f} bar)")
ax2.set_xlabel("Surface temperature [K]")
ax2.set_ylabel("Surface pressure [bar]")
ax2.set_title("Surface pressure")
ax2.legend()

plt.show()

print(results[["T_surface", "M_total_over_M_planet", "P_surface_bar",
               "mass_frac_gas", "mass_frac_silicate", "mass_frac_metal"]].to_string(index=False))

In [ ]:
results.to_csv(cfg.run_dir / "grid_summary.csv", index=False)